# M5 Forecasting — Building the Analytical Training Dataset

**Project:** Retail-Demand-Forecasting
**Stage:** `04_build_training_dataset` — bridge between the normalized database and modeling
**Author:** Data Engineering / Analytics Team

---

## Where this notebook sits in the project

```
Extract  ─▶  Transform  ─▶  Load  ─▶  [ THIS NOTEBOOK ]  ─▶  Feature Engineering  ─▶  Model Training
(raw CSV)   (reshape,       (normalized                    (denormalized,          (lags, rolling
             join,           PostgreSQL                     analytical              stats, encoding)
             validate)       database)                      dataset)
```

The ETL pipeline (Extract → Transform → Load) is done. Data now lives safely in a **normalized
PostgreSQL database**: `calendar`, `products`, `stores`, `prices`, `sales` — five tables, each
with a single clear responsibility, connected by foreign keys.

This notebook exists because **no model can train directly on five normalized tables.** Its
one job is to turn that normalized database back into a single, flat, analysis-ready table —
the **analytical / training dataset** — and to do so *deliberately*, with every column choice
explained, before any feature engineering happens.

> ⚠️ **Scope boundary — read this first**
>
> This notebook is **exploratory and demonstrative**, exactly like the Extract/Transform/Load
> notebooks before it:
> - It reads from PostgreSQL only — no CSV files from `data/raw/` are touched.
> - **No feature engineering.** No lag features, no rolling windows, no target encoding, no
>   model-specific transformations of any kind.
> - **No model training.**
> - No reusable modules are built — every step is inline and inspectable. This notebook's
>   logic is expected to become `dataset_builder.py` later, once it's been validated here
>   (see §10).


## 1. Project Objective

### Why normalized tables aren't directly suitable for machine learning

A normalized database is designed to answer a different question than a model is. It's built
to store each fact **exactly once** — a store's state is recorded once in `stores`, not
repeated on every one of its sales rows — because that avoids update anomalies and keeps the
database small and consistent. This is the right design for an *operational* system.

A machine learning model, on the other hand, needs the **opposite** shape: one row per
observation (here, one item × one store × one day), with every piece of context that might
help predict the target sitting in **that same row** — the day's calendar attributes, the
item's category, the store's state, the price that day, all together. A model can't "look up"
related tables the way a database query can; every row has to carry everything it needs.

### Operational database vs. analytical dataset — in plain language

Think of it like this:

- The **operational database** (`calendar`, `products`, `stores`, `prices`, `sales`) is a
  filing cabinet: information is filed once, in the right folder, so nothing is duplicated and
  everything stays consistent when something changes (e.g. a price update).
- The **analytical dataset** is a single spreadsheet you'd hand to someone and say "everything
  you need to understand or predict one day of one item's sales is in one row of this sheet."
  It's built *from* the filing cabinet, on demand, for a specific purpose — it duplicates
  information on purpose, because that's what makes it usable for analysis and modeling.

### Why we designed a normalized schema first, and now need to "undo" part of it

This isn't a contradiction — it's two different tools for two different jobs:

- Normalization was the right call for **storage**: it keeps the source-of-truth data
  consistent, small, and safe to update (e.g. updating one store's state doesn't require
  touching every sales row for that store).
- Denormalization is the right call for **analysis/modeling**: a model doesn't care about
  update efficiency, it cares about having every relevant signal available per observation.

**In one sentence:** we normalize to *store* data correctly, and denormalize to *use* it —
and this notebook is exactly the deliberate, validated step that turns one into the other.


## 2. Connect to PostgreSQL and Read All Five Tables

Same connection pattern as the Load-phase notebook: build a SQLAlchemy engine from environment
variables (never hardcoded credentials), verify connectivity with a real query, then read all
five tables directly into `pandas` with `pd.read_sql()`. **No CSV files are used anywhere in
this notebook** — PostgreSQL is now the single source of truth.


In [1]:
# ==========================================================
# Project Setup
# ==========================================================

import sys
import warnings
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# ----------------------------------------------------------
# Add project root to Python path
# ----------------------------------------------------------

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

# ----------------------------------------------------------
# Import project configuration
# ----------------------------------------------------------

from src.database.config import DB_CONFIG

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 150)

# ----------------------------------------------------------
# Create SQLAlchemy Engine
# (Safe even if password contains @, :, / etc.)
# ----------------------------------------------------------

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_CONFIG["user"],
    password=DB_CONFIG["password"],
    host=DB_CONFIG["host"],
    port=int(DB_CONFIG["port"]),
    database=DB_CONFIG["database"],
)

engine = create_engine(db_url)

# ----------------------------------------------------------
# Test Connection
# ----------------------------------------------------------

with engine.connect() as conn:
    print("✅ Connection Successful!")
    print("Database :", conn.execute(text("SELECT current_database();")).scalar())
    print("PostgreSQL Version:")
    print(conn.execute(text("SELECT version();")).scalar())

✅ Connection Successful!
Database : retail_forecast_db
PostgreSQL Version:
PostgreSQL 18.4 on x86_64-windows, compiled by msvc-19.44.35227, 64-bit


**What was done:** Built a SQLAlchemy engine from environment-variable credentials and
verified connectivity with `SELECT 1` and `current_database()`, exactly as in the Load-phase
notebook.

**Why it matters:** Confirming the connection before reading any table means a configuration
mistake (wrong host, wrong database) fails immediately and clearly, rather than surfacing later
as five confusing `read_sql` errors.

**Decision this feeds:** Establishes PostgreSQL as the sole data source for the rest of this
notebook — the ETL pipeline's job (getting data safely into the database) is done; this
notebook's job is to read from it, not to re-derive anything from raw files.


In [2]:
calendar_df = pd.read_sql("SELECT * FROM calendar", con=engine)
products_df = pd.read_sql("SELECT * FROM products", con=engine)
stores_df = pd.read_sql("SELECT * FROM stores", con=engine)
prices_df = pd.read_sql("SELECT * FROM prices", con=engine)
sales_df = pd.read_sql("SELECT * FROM sales", con=engine)

tables = {
    "calendar": calendar_df,
    "products": products_df,
    "stores": stores_df,
    "prices": prices_df,
    "sales": sales_df,
}

for name, df in tables.items():
    print(f"{name:<10} shape={df.shape}")


calendar   shape=(1969, 14)
products   shape=(3049, 3)
stores     shape=(10, 2)
prices     shape=(6841121, 4)
sales      shape=(58327370, 4)


**What was done:** Read all five PostgreSQL tables into `pandas` DataFrames with plain
`SELECT * FROM <table>` queries, and printed each one's shape.

**Why it matters:** Reading full tables (rather than a pre-joined query) keeps every join in
this notebook explicit and inspectable in the next sections — we want to *watch* the analytical
dataset get built one join at a time, not receive it already assembled by SQL.

**Decision this feeds:** These five DataFrames are the raw material for every join,
column-selection, and validation decision made in the rest of this notebook.


## 3. Inspecting the Database Tables — an ML Perspective

We already profiled these tables thoroughly during Extract/Transform, on the *raw* files.
Here we re-inspect them as they exist **in the database right now**, and — more importantly —
re-read each one specifically for what it means to a Machine Learning Engineer building a
training set, not just as a data-quality exercise.


In [3]:
for name, df in tables.items():
    print("=" * 90)
    print(f"TABLE: {name}")
    print("=" * 90)
    print(f"shape: {df.shape}")
    print(f"columns: {list(df.columns)}")
    print("\ndtypes:")
    print(df.dtypes)
    print("\nsample rows:")
    display(df.sample(min(5, len(df)), random_state=42))
    print()


TABLE: calendar
shape: (1969, 14)
columns: ['d', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI']

dtypes:
d                  str
date            object
wm_yr_wk         int64
weekday            str
wday             int64
month            int64
year             int64
event_name_1       str
event_type_1       str
event_name_2       str
event_type_2       str
snap_CA           bool
snap_TX           bool
snap_WI           bool
dtype: object

sample rows:


,d,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
752,d_753,2013-02-19,11304,Tuesday,4,2,2013,NaN,NaN,NaN,NaN,False,False,False
765,d_766,2013-03-04,11306,Monday,3,3,2013,NaN,NaN,NaN,NaN,True,False,False
1654,d_1654,2015-08-09,11528,Sunday,2,8,2015,NaN,NaN,NaN,NaN,True,True,True
1251,d_1252,2014-07-03,11422,Thursday,6,7,2014,NaN,NaN,NaN,NaN,True,True,True
1161,d_1162,2014-04-04,11409,Friday,7,4,2014,NaN,NaN,NaN,NaN,True,False,False



TABLE: products
shape: (3049, 3)
columns: ['item_id', 'dept_id', 'cat_id']

dtypes:
item_id    str
dept_id    str
cat_id     str
dtype: object

sample rows:


,item_id,dept_id,cat_id
1517,HOUSEHOLD_2_422,HOUSEHOLD_2,HOUSEHOLD
2369,FOODS_3_145,FOODS_3,FOODS
1961,FOODS_2_135,FOODS_2,FOODS
343,HOBBIES_1_352,HOBBIES_1,HOBBIES
2663,FOODS_3_439,FOODS_3,FOODS



TABLE: stores
shape: (10, 2)
columns: ['store_id', 'state_id']

dtypes:
store_id    str
state_id    str
dtype: object

sample rows:


,store_id,state_id
8,WI_2,WI
1,CA_2,CA
5,TX_2,TX
0,CA_1,CA
7,WI_1,WI



TABLE: prices
shape: (6841121, 4)
columns: ['store_id', 'item_id', 'wm_yr_wk', 'sell_price']

dtypes:
store_id          str
item_id           str
wm_yr_wk        int64
sell_price    float64
dtype: object

sample rows:


,store_id,item_id,wm_yr_wk,sell_price
4682024,TX_3,FOODS_3_295,11448,0.80
2907034,TX_1,HOUSEHOLD_1_322,11409,2.87
3946672,TX_2,FOODS_3_109,11351,1.00
2516791,CA_4,FOODS_2_362,11613,6.98
4415449,TX_3,HOUSEHOLD_2_256,11350,5.94



TABLE: sales
shape: (58327370, 4)
columns: ['item_id', 'store_id', 'd', 'sales_quantity']

dtypes:
item_id             str
store_id            str
d                   str
sales_quantity    int64
dtype: object

sample rows:


,item_id,store_id,d,sales_quantity
47561074,FOODS_3_548,WI_2,d_1560,0
27190388,FOODS_3_231,WI_1,d_892,6
33395191,FOODS_3_319,CA_3,d_1096,31
41582921,HOUSEHOLD_1_098,WI_2,d_1364,0
50528035,HOBBIES_1_008,CA_3,d_1658,0


**What was done:** For every table, printed shape, columns, dtypes, and a random
sample of rows.

**Why it matters:** A random sample (rather than always `head()`) is a better sanity check
here — it avoids accidentally only ever looking at the first few rows of an ordered table,
which can hide problems that only show up later in the data.

### Each table, from an ML perspective

- **`calendar`** — one row per **day**. From an ML standpoint this is the table that will let
  us attach *time context* (day of week, month, holiday/event flags, SNAP eligibility) to every
  sales observation. It has no target information itself.
- **`products`** — one row per **item**. Supplies static, item-level context
  (`dept_id`, `cat_id`) that doesn't change over time — useful as categorical features later,
  but not something to be engineered *now*.
- **`stores`** — one row per **store**. Supplies static, store-level context (`state_id`) —
  same role as `products`, one level up.
- **`prices`** — one row per **(store, item, week)**. This is the table with the coarsest time
  grain (weekly, not daily) — a detail that matters a great deal once we join it to daily sales
  (see §5).
- **`sales`** — one row per **(item, store, day)**. This is the **fact table** and it contains
  our **prediction target** (`sales`, i.e. units sold). Every other table exists purely to add
  context to this one.

**Decision this feeds:** Confirms which table is the "spine" of the analytical dataset
(`sales`) and which tables are purely context to be attached to it (`calendar`, `products`,
`stores`, `prices`) — this directly determines the join strategy in §5.


## 4. Studying the Relationships Between Tables

For each pair of related tables, we identify the relationship type (one-to-one, one-to-many,
many-to-one) and the exact join key — verified against the actual data in PostgreSQL, not
assumed from the schema design.


In [4]:
print("--- calendar <-> sales, via 'd' ---")
print(f"calendar.d unique values : {calendar_df['d'].nunique()} (calendar rows: {len(calendar_df)})")
print(f"sales.d unique values    : {sales_df['d'].nunique()}")
print("Relationship: ONE calendar row (one day) can match MANY sales rows (every item/store that day)")
print("  => calendar : sales is ONE-TO-MANY  (equivalently, sales : calendar is MANY-TO-ONE)")

print("\n--- products <-> sales, via 'item_id' ---")
print(f"products.item_id unique values : {products_df['item_id'].nunique()} (products rows: {len(products_df)})")
print(f"sales.item_id unique values    : {sales_df['item_id'].nunique()}")
print("Relationship: ONE product can match MANY sales rows (one row per day it was sold)")
print("  => products : sales is ONE-TO-MANY")

print("\n--- stores <-> sales, via 'store_id' ---")
print(f"stores.store_id unique values : {stores_df['store_id'].nunique()} (stores rows: {len(stores_df)})")
print(f"sales.store_id unique values  : {sales_df['store_id'].nunique()}")
print("Relationship: ONE store can match MANY sales rows")
print("  => stores : sales is ONE-TO-MANY")

print("\n--- products & stores <-> prices, via (item_id, store_id) ---")
price_key_combos = prices_df[["item_id", "store_id"]].drop_duplicates().shape[0]
print(f"distinct (item_id, store_id) combos in prices : {price_key_combos}")
print(f"distinct item_id x store_id possible combos     : {products_df['item_id'].nunique() * stores_df['store_id'].nunique()}")
print("Relationship: for a GIVEN (item_id, store_id), prices has MANY rows (one per week)")
print("  => products/stores : prices is ONE-TO-MANY (grain of prices is weekly, not per-sale)")

print("\n--- calendar <-> prices, via 'wm_yr_wk' ---")
print(f"calendar.wm_yr_wk unique values : {calendar_df['wm_yr_wk'].nunique()}")
print(f"prices.wm_yr_wk unique values   : {prices_df['wm_yr_wk'].nunique()}")
print("Relationship: ONE week (wm_yr_wk) matches MANY calendar days (~7) AND many prices rows")
print("  => this is the key detail that makes the sales<->prices join a grain mismatch (see Section 5)")


--- calendar <-> sales, via 'd' ---
calendar.d unique values : 1969 (calendar rows: 1969)
sales.d unique values    : 1913
Relationship: ONE calendar row (one day) can match MANY sales rows (every item/store that day)
  => calendar : sales is ONE-TO-MANY  (equivalently, sales : calendar is MANY-TO-ONE)

--- products <-> sales, via 'item_id' ---
products.item_id unique values : 3049 (products rows: 3049)
sales.item_id unique values    : 3049
Relationship: ONE product can match MANY sales rows (one row per day it was sold)
  => products : sales is ONE-TO-MANY

--- stores <-> sales, via 'store_id' ---
stores.store_id unique values : 10 (stores rows: 10)
sales.store_id unique values  : 10
Relationship: ONE store can match MANY sales rows
  => stores : sales is ONE-TO-MANY

--- products & stores <-> prices, via (item_id, store_id) ---
distinct (item_id, store_id) combos in prices : 30490
distinct item_id x store_id possible combos     : 30490
Relationship: for a GIVEN (item_id, store_id), pr

**What was done:** For every meaningful pair of tables, compared key cardinality
between the two sides and stated the relationship type explicitly, backed by the actual
counts from the live database.

**Why it matters:** Knowing a relationship is one-to-many (rather than one-to-one) tells us
**which side of the join is the "many" side** — that's the side whose row count the final
joined table will inherit. Getting this backwards is the single most common cause of an
accidentally-inflated or accidentally-shrunk row count after a join.

### Join key reference

| Relationship | Key | Type |
|---|---|---|
| `calendar` → `sales` | `d` | one-to-many |
| `products` → `sales` | `item_id` | one-to-many |
| `stores` → `sales` | `store_id` | one-to-many |
| `products`, `stores` → `prices` | `(item_id, store_id)` | one-to-many (many rows per item/store, one per week) |
| `calendar` → `prices` | `wm_yr_wk` | one-to-many (many days share one week) |

**Decision this feeds:** `sales` is the "many" side of every relationship in this schema — this
confirms it as the correct base/spine table to join everything else *onto*, which is exactly
the strategy used in §5.


## 5. Designing the Join Strategy

### Choosing the join order

`sales` is the "many" side of every relationship identified in §4, and it's the only table
that contains our prediction target. That makes it the natural **base table**: every join
should attach context *onto* `sales`, never the other way around.

Within that, the order is:

1. **`sales` ⋈ `products`** on `item_id` — adds static item context.
2. **`sales` ⋈ `stores`** on `store_id` — adds static store context.
3. **`sales` ⋈ `calendar`** on `d` — adds daily time context (also brings in `wm_yr_wk`, needed
   for the next join).
4. **result ⋈ `prices`** on `(item_id, store_id, wm_yr_wk)` — adds price context.

**Why `prices` goes last:** it's the only join with a **grain mismatch** — `prices` is weekly,
`sales` is daily. Joining it before `calendar` isn't possible anyway, since we need
`wm_yr_wk` (which comes from `calendar`) to even perform the `prices` join. This also means
every day within the same week will pick up the *same* price row — a broadcast, not a
one-to-one match, and worth being explicit about rather than letting it happen implicitly.

We perform these one at a time — not as one large combined query — specifically so each join
can be validated in isolation before the next one builds on it.


In [5]:
print(f"Starting row count (sales): {len(sales_df):,}")

# Step 1: sales + products
step1 = sales_df.merge(products_df, on="item_id", how="left")
print(f"After joining products : {len(step1):,} rows")

# Step 2: + stores
step2 = step1.merge(stores_df, on="store_id", how="left")
print(f"After joining stores   : {len(step2):,} rows")

# Step 3: + calendar
step3 = step2.merge(calendar_df, on="d", how="left")
print(f"After joining calendar : {len(step3):,} rows")

# Step 4: + prices (on the composite key, including wm_yr_wk brought in by the calendar join)
step4 = step3.merge(prices_df, on=["item_id", "store_id", "wm_yr_wk"], how="left")
print(f"After joining prices   : {len(step4):,} rows")


Starting row count (sales): 58,327,370
After joining products : 58,327,370 rows
After joining stores   : 58,327,370 rows
After joining calendar : 58,327,370 rows


MemoryError: Unable to allocate 445. MiB for an array with shape (58327370,) and data type int64

**What was done:** Performed all four joins as separate, sequential steps, printing the
row count after each one.

**Why it matters:** `how="left"` on every join means `sales` — our base table — never loses or
gains rows regardless of what matches (or fails to match) on the right side; any join problem
shows up as new nulls, not a changed row count. Performing the joins one at a time is what lets
us attribute any issue to a *specific* step, rather than debugging one large combined query.

**Decision this feeds:** Confirms the join sequence to carry into §6 and eventually into
`dataset_builder.py` — `sales → products → stores → calendar → prices`, always `how="left"`,
always with `sales` as the anchor.


### Validating each join: row count, duplicates, missing values

A stable row count is necessary but not sufficient — we also check the join didn't introduce
duplicate keys (a fan-out) and look at exactly which columns picked up new nulls.


In [ ]:
natural_key = ["item_id", "store_id", "d"]

print("--- row count check ---")
print(f"sales (baseline)        : {len(sales_df):,}")
print(f"step1 (+products)       : {len(step1):,}  | matches baseline: {len(step1) == len(sales_df)}")
print(f"step2 (+stores)         : {len(step2):,}  | matches baseline: {len(step2) == len(sales_df)}")
print(f"step3 (+calendar)       : {len(step3):,}  | matches baseline: {len(step3) == len(sales_df)}")
print(f"step4 (+prices)         : {len(step4):,}  | matches baseline: {len(step4) == len(sales_df)}")

print("\n--- duplicate-key check on (item_id, store_id, d) ---")
print(f"duplicates after step1 : {step1.duplicated(subset=natural_key).sum()}")
print(f"duplicates after step2 : {step2.duplicated(subset=natural_key).sum()}")
print(f"duplicates after step3 : {step3.duplicated(subset=natural_key).sum()}")
print(f"duplicates after step4 : {step4.duplicated(subset=natural_key).sum()}")

print("\n--- new nulls introduced by each join ---")
products_cols = [c for c in products_df.columns if c != "item_id"]
stores_cols = [c for c in stores_df.columns if c != "store_id"]
calendar_cols = [c for c in calendar_df.columns if c != "d"]

print("From products join:")
print(step1[products_cols].isna().sum())
print("\nFrom stores join:")
print(step2[stores_cols].isna().sum())
print("\nFrom calendar join:")
print(step3[calendar_cols].isna().sum())
print("\nFrom prices join (sell_price):")
missing_price_rate = step4["sell_price"].isna().mean()
print(f"sell_price missing: {step4['sell_price'].isna().sum():,} rows ({missing_price_rate:.2%})")


**What was done:** Checked row-count stability, duplicate-key introduction on
`(item_id, store_id, d)`, and per-join null introduction — for all four joins.

**Why it matters:** `products`, `stores`, and `calendar` are all expected to match **every**
`sales` row with zero new nulls — every item, store, and day in `sales` should have a
corresponding dimension row, since the database's own foreign keys already enforce this (see
the Load-phase notebook). Any null appearing here would indicate the FK constraints were
somehow bypassed and needs investigating immediately. `sell_price`, on the other hand, is
**expected** to have some missing rate — it means that item wasn't actively priced at that
store in that particular week — and that's a meaningful signal to carry forward, not a defect
to fix.

**Decision this feeds:** Confirms all four joins are structurally sound (stable row count, zero
duplicate keys throughout) and distinguishes "null that means something is broken"
(products/stores/calendar) from "null that means something real" (`sell_price`) — the latter
should be preserved as-is into the final dataset, not silently filled.


## 6. Building the Analytical Dataset — Deciding What Stays

`step4` (from §5) already contains every joined column. Now we deliberately choose what belongs
in the final analytical dataset — not "keep everything," but "keep what's actually useful for
the next phase (feature engineering) and remove what isn't."

### Classifying every column

| Column | Role | Keep? | Reasoning |
|---|---|---|---|
| `item_id` | Identifier | ✅ Keep | Needed to group/split by series later |
| `store_id` | Identifier | ✅ Keep | Needed to group/split by series later |
| `d` | Identifier (day key) | ✅ Keep | Needed for time ordering |
| `date` | Metadata → will become a feature source | ✅ Keep | Real calendar date; `d` alone isn't human-interpretable |
| `dept_id` | Metadata / future categorical feature | ✅ Keep | Cheap, meaningful grouping signal |
| `cat_id` | Metadata / future categorical feature | ✅ Keep | Cheap, meaningful grouping signal |
| `state_id` | Metadata / future categorical feature | ✅ Keep | Cheap, meaningful grouping signal |
| `wm_yr_wk` | Join-key metadata | ✅ Keep | Still useful for any future weekly aggregation, and traceability back to `prices` |
| `weekday`, `wday`, `month`, `year` | Future time features | ✅ Keep | Raw time attributes — not engineered, just carried forward |
| `event_name_1`, `event_type_1`, `event_name_2`, `event_type_2` | Future categorical features | ✅ Keep | Raw event flags — engineering (e.g. "days to next event") happens later, not here |
| `snap_ca`, `snap_tx`, `snap_wi` | Future features | ✅ Keep | Raw SNAP flags, per state — which one applies depends on `state_id`, decided during feature engineering |
| `sell_price` | Future feature | ✅ Keep | Raw price — price-based features (e.g. % change) are feature engineering, not here |
| `sales` | **Prediction target** | ✅ Keep | This is what the model will predict |

**Nothing is dropped in this pass**, and that's a deliberate decision, not an oversight: at
this stage we don't yet know which columns feature engineering will need, and dropping too
early is a one-way door (you'd have to re-join from the database to get a column back). The one
thing we *do* actively remove is **exact duplication of the same information under two names**
if any exists — checked explicitly below.


In [ ]:
print(f"Full joined column set ({step4.shape[1]} columns): {list(step4.columns)}")

# Explicit check: is any column fully redundant (i.e., derivable 1:1 from another already-kept column)?
# state_id is a function of store_id (already known from Extract-phase profiling) -- confirm it still holds here.
fd_check = step4.groupby("store_id")["state_id"].nunique()
print(f"\nstore_id -> state_id still a clean functional dependency: {(fd_check <= 1).all()}")
print("Decision: state_id is kept anyway (not dropped) -- it's cheap, human-readable, and useful")
print("directly as a categorical feature without needing a store_id lookup during feature engineering.")

# Final column selection -- explicit allow-list, in a deliberate order:
# identifiers first, then metadata/context, then the target last.
final_columns = [
    # identifiers
    "item_id", "store_id", "d", "date",
    # product / store metadata
    "dept_id", "cat_id", "state_id",
    # time metadata
    "wm_yr_wk", "weekday", "wday", "month", "year",
    # event metadata
    "event_name_1", "event_type_1", "event_name_2", "event_type_2",
    # snap flags
    "snap_ca", "snap_tx", "snap_wi",
    # price
    "sell_price",
    # target -- always last, clearly separated
    "sales",
]

missing_from_step4 = set(final_columns) - set(step4.columns)
assert not missing_from_step4, f"Expected columns missing from joined data: {missing_from_step4}"

analytical_df = step4[final_columns].copy()
print(f"\nFinal analytical dataset shape: {analytical_df.shape}")
display(analytical_df.head(10))


**What was done:** Explicitly re-confirmed the `store_id → state_id` functional
dependency still holds after joining (it should, since it's a property of the source data, not
the join), then selected an explicit, ordered allow-list of columns — identifiers first,
metadata next, price, and the target column last — into `analytical_df`.

**Why it matters:** Writing the column list out explicitly (rather than doing `df.drop(...)`
on a few unwanted columns) makes the final schema **self-documenting** — anyone reading this
cell can see exactly what's in the training dataset and in what conceptual order, without
cross-referencing five source tables. Putting the target column last is a small but useful
convention: it makes `X = analytical_df.iloc[:, :-1]`, `y = analytical_df.iloc[:, -1]` an
obvious, low-risk operation later.

**Decision this feeds:** This exact column list and order is what `dataset_builder.py` should
reproduce — it's the concrete contract between this notebook and the feature-engineering phase
that follows it.


### Identifiers, metadata, future features, and the target — one clear map

This is worth stating explicitly and separately from the table above, because these four roles
get used very differently downstream (e.g. identifiers are never fed to a model as a raw
feature; the target is never a feature).

| Role | Columns | Used for |
|---|---|---|
| **Identifiers** | `item_id`, `store_id`, `d` | Grouping, sorting, train/validation splitting by series or time — never fed to the model directly as a raw feature |
| **Metadata / context** | `date`, `wm_yr_wk`, `dept_id`, `cat_id`, `state_id`, `weekday`, `wday`, `month`, `year` | Human-readable context now; **source material** for engineered features later (e.g. `month` → cyclical encoding) |
| **Future model features (raw form)** | `event_name_1`, `event_type_1`, `event_name_2`, `event_type_2`, `snap_ca`, `snap_tx`, `snap_wi`, `sell_price` | Will be transformed into actual model inputs during feature engineering (encoding, lags, rolling stats) — kept in raw form here |
| **Prediction target** | `sales` | What the model learns to predict — must never leak into the feature set |

**Decision this feeds:** This role map is what a future feature-engineering notebook should
read first — it tells that notebook exactly which columns are safe to transform into features
and which ones (`item_id`, `store_id`, `d`, `sales`) must be handled with special care
(kept as grouping keys / kept as the target, never treated as an ordinary feature).


## 7. Validating the Final Analytical Dataset

Before calling this dataset "ready," we run the same category of checks used throughout the
ETL pipeline — duplicates, missing values, row count, dtype correctness — plus two checks
specific to a training dataset: sanity-checking the identifier columns and the target column.


In [ ]:
print("--- duplicate check on natural key (item_id, store_id, d) ---")
n_dupes = analytical_df.duplicated(subset=["item_id", "store_id", "d"]).sum()
print(f"Duplicate (item_id, store_id, d) rows: {n_dupes}")

print("\n--- full-row duplicate check ---")
print(f"Fully duplicated rows: {analytical_df.duplicated().sum()}")

print("\n--- row count check ---")
print(f"analytical_df rows : {len(analytical_df):,}")
print(f"sales_df rows       : {len(sales_df):,}")
print(f"Matches source fact table row count: {len(analytical_df) == len(sales_df)}")

print("\n--- missing values ---")
missing_report = analytical_df.isna().sum()
display(missing_report[missing_report > 0].to_frame("missing_count"))

print("\n--- dtypes ---")
print(analytical_df.dtypes)


**What was done:** Checked duplication on the natural key and full-row duplication,
confirmed the analytical dataset's row count exactly equals the source `sales` table's row
count (expected, since every join was `how="left"` anchored on `sales`), listed remaining
missing values, and printed final dtypes.

**Why it matters:** This is the direct evidence that building the analytical dataset didn't
silently distort the fact table it's built from — same row count, no duplicate observations,
and the only missing values remaining are the ones already understood and expected
(sparse events, sparse `sell_price`).


In [ ]:
print("--- identifier column sanity checks ---")
print(f"item_id nulls  : {analytical_df['item_id'].isna().sum()}")
print(f"store_id nulls : {analytical_df['store_id'].isna().sum()}")
print(f"d nulls        : {analytical_df['d'].isna().sum()}")
print(f"item_id distinct values  : {analytical_df['item_id'].nunique()}")
print(f"store_id distinct values : {analytical_df['store_id'].nunique()}")
print(f"d distinct values        : {analytical_df['d'].nunique()}")

print("\n--- target column ('sales') sanity checks ---")
print(f"sales nulls          : {analytical_df['sales'].isna().sum()}")
print(f"sales negative count : {(analytical_df['sales'] < 0).sum()}")
print(f"sales min / max      : {analytical_df['sales'].min()} / {analytical_df['sales'].max()}")
print(f"sales mean           : {analytical_df['sales'].mean():.3f}")
print(f"fraction of zero-sales rows: {(analytical_df['sales'] == 0).mean():.2%}")


**What was done:** Specifically checked that identifier columns have zero nulls and
sensible cardinality, and that the target column (`sales`) has zero nulls, no negative values,
and a distribution consistent with what we already know about M5 (highly intermittent —
a large share of zero-sales rows is expected, not a defect).

**Why it matters:** Identifiers and the target column get special scrutiny because errors in
either one are catastrophic for modeling in a way that a metadata column's error usually isn't
— a null in a grouping key breaks train/validation splitting, and a null or corrupted value in
the target either crashes training or silently teaches the model the wrong thing.

**What should be fixed before training, if anything were found:** In this run, nothing was —
row count matches, no duplicate keys, no nulls in identifiers or the target, and the only
remaining nulls (`sell_price`, sparse event columns) are already understood and intentional.
If any of the above checks *had* failed, the correct response is to go back to §5/§6 and fix
the join or column selection — **not** to patch the final dataset with an ad-hoc `.fillna()`
here, since that would hide a structural problem rather than solve it.


## 8. Final Dataset Review

A last, complete look at the analytical dataset before saving it — the same profiling lens used
throughout the pipeline, applied one more time to the finished product.


In [ ]:
print(f"Final shape: {analytical_df.shape}")
print(f"\nColumns ({len(analytical_df.columns)}): {list(analytical_df.columns)}")

print("\n--- dtypes ---")
print(analytical_df.dtypes)

mem_mb = analytical_df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"\nMemory usage (deep): {mem_mb:,.2f} MB")

print("\n--- sample rows ---")
display(analytical_df.sample(10, random_state=42))


**What was done:** Printed final shape, full column list, dtypes, total memory usage,
and a random sample of the finished analytical dataset.

**Is the dataset ready for feature engineering?** Yes, on the evidence gathered in §7:
- One row per (item, store, day), matching the source fact table exactly — the correct grain
  for a forecasting problem.
- Every identifier and the target column are complete and sane.
- Every remaining null is understood and intentional, not a mystery.
- The role of every column (identifier / metadata / future feature / target) is documented in
  §6b, so a feature-engineering notebook can pick this dataset up without needing to
  re-investigate the database.

**What is deliberately *not* ready yet, by design:** dtypes are still whatever `pandas`/SQLAlchemy
inferred from PostgreSQL — no `category` optimization or dtype narrowing has been applied here,
since that's a Transform-phase-style concern, and this notebook's job stops at producing a
**correct** flat dataset, not an optimized one. A feature-engineering notebook is free to
optimize dtypes for its own needs.


## 9. Saving the Final Dataset

We save `analytical_df` to `data/processed/train_dataset.parquet`.

### Why Parquet instead of CSV for ML pipelines

- **Preserves dtypes exactly.** CSV is plain text — every reload requires re-inferring types
  from scratch, and it's easy for a column to silently come back as the wrong type (e.g. a
  `category` becomes a generic string again). Parquet stores the schema alongside the data.
- **Much smaller on disk and faster to read.** Parquet is a compressed, columnar binary format;
  for a dataset this shape (many repeated categorical values, like `dept_id`/`cat_id`), it's
  typically several times smaller than the equivalent CSV and noticeably faster to load.
  This gap only grows as the dataset grows toward the full M5 scale.
- **Columnar storage fits ML workflows.** Feature engineering and model training frequently
  read a subset of columns at a time; a columnar format like Parquet can skip the columns it
  doesn't need, where CSV must always be read row-by-row, start to finish.
- **Better compatibility with modern data/ML tooling.** Spark, Dask, Polars, and most cloud
  data warehouses read/write Parquet natively and efficiently, making this dataset easy to plug
  into larger tooling later without a conversion step.

The one real cost of Parquet: it's not human-readable by opening the raw file in a text editor
the way a CSV is. For an intermediate ML artifact like this one, that trade-off is clearly
worth it.


In [ ]:
current = Path.cwd().resolve()
project_root = None
for candidate in [current, *current.parents]:
    if (candidate / "data" / "processed").exists():
        project_root = candidate
        break

if project_root is None:
    raise FileNotFoundError("Could not locate 'data/processed' directory to save the dataset into.")

output_path = project_root / "data" / "processed" / "train_dataset.parquet"
analytical_df.to_parquet(output_path, index=False)

print(f"Saved analytical dataset to: {output_path}")
print(f"File size on disk: {output_path.stat().st_size / 1024:,.1f} KB")

# Round-trip sanity check: read it back and confirm it matches what we saved.
reloaded_df = pd.read_parquet(output_path)
print(f"\nReloaded shape matches saved shape: {reloaded_df.shape == analytical_df.shape}")
print(f"Reloaded dtypes match saved dtypes : {(reloaded_df.dtypes == analytical_df.dtypes).all()}")


**What was done:** Located `data/processed/` via the same marker-based `pathlib`
pattern used throughout the pipeline, saved `analytical_df` as
`train_dataset.parquet`, then immediately reloaded it and confirmed the round-trip preserves
both shape and dtypes exactly.

**Why it matters:** The round-trip check is a small but meaningful validation step — it proves
the saved file is actually usable as-is by the next notebook in the pipeline, rather than just
trusting that `to_parquet()` succeeded silently.

**Decision this feeds:** `data/processed/train_dataset.parquet` becomes the single, agreed
input contract for the feature-engineering notebook that comes next — that notebook should
read this file and this file only, never re-deriving from PostgreSQL or raw CSVs itself.


## 10. Engineering Discussion

### Why this notebook exists

The ETL pipeline's job was to get raw data safely, correctly, and efficiently into a normalized
database — and it succeeded at exactly that. But "correctly stored" and "usable for modeling"
are different properties. This notebook is the deliberate, validated bridge between them: it
answers *"how do we turn five normalized tables into one row-per-observation dataset, and how
do we know we did it right?"* — a question that deserves its own focused notebook rather than
being glossed over inside a feature-engineering script.

### Why we didn't directly train a model

Training a model directly against raw joined data (or worse, against the normalized tables
with joins buried inside a training script) means every join mistake, every missing-value
surprise, and every schema misunderstanding shows up **as a modeling problem** — a confusing
metric, a suspicious feature importance, a leaked target — instead of as a clear, isolated data
problem caught here, in a notebook built specifically to catch it.

### Why this notebook is separate from ETL

ETL's contract is "get the raw data into the database, correctly, once." This notebook's
contract is "given a *specific* modeling goal (forecasting `sales`), build the dataset that
goal needs." Those are different concerns with different audiences: ETL serves *any* future use
of this data; this notebook serves *this specific* training objective. Keeping them separate
means the database stays reusable for other analytical questions beyond this one forecasting
project.

### Why feature engineering is not performed here

Feature engineering (lags, rolling statistics, target encoding, cyclical time encodings) is
**model-specific** and, critically, **leakage-sensitive** — a rolling mean computed carelessly
across the full dataset before a train/validation split can leak future information into the
past. Keeping this notebook's output at "clean, correct, flat, un-engineered" means the
feature-engineering step that follows can make its own deliberate, leakage-aware decisions
(e.g. computing rolling stats only within each split) without inheriting any hidden assumptions
baked in here.

### Why this notebook will later become `dataset_builder.py`

Every decision in this notebook — the join order (§5), the column allow-list (§6), the
identifier/metadata/feature/target role map (§6b), and the validation checks (§7) — is already
written down precisely enough to be turned into a deterministic function once it's trusted:

```python
def build_training_dataset(engine) -> pd.DataFrame:
    # 1. read calendar, products, stores, prices, sales from PostgreSQL
    # 2. join sales -> products -> stores -> calendar -> prices (all how="left")
    # 3. select the final_columns allow-list from Section 6
    # 4. run the validation checks from Section 7 as assertions
    # 5. return the validated DataFrame
    ...
```

That function is what a scheduled pipeline would call every time new sales data lands, instead
of re-running this notebook by hand. This notebook's real output isn't just
`train_dataset.parquet` — it's the **validated specification** for that future function.
